In [1]:
# ==============================
# 1. IMPORTS + SETTINGS
# ==============================
from pathlib import Path
import numpy as np
import pandas as pd

# ---- CHANGE ONLY THESE PATHS IF NEEDED ----
INPUT_FILE = Path(r"Conversion_data_weekly.csv")
OUTPUT_CSV = Path(r"conversion_data_detailed.csv")
OUTPUT_XLSX = Path(r"conversion_data_detailed.xlsx")

# Segment grain. Change here only if your column names differ.
GROUP_COLS = ["VRLOCATION", "LANE"]

# A +/- 5 percentage-point WoW move is treated as stable.
TREND_THRESHOLD = 0.05

# Minimum historical observations needed before calculating a rolling metric.
# 1 means early weeks can still receive a partial-history average.
MIN_PERIODS = 1


In [2]:
# ==============================
# 2. ROBUST CSV READER
# ==============================
def read_csv_robust(path):
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    last_error = None
    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            print(f"Loaded {path.name} using encoding: {enc}")
            return df
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error

df = read_csv_robust(INPUT_FILE)
print(f"Rows: {len(df):,} | Columns: {len(df.columns):,}")
display(df.head())


Loaded Conversion_data_weekly.csv using encoding: cp1252
Rows: 1,714,155 | Columns: 17


,SALEDATE_Year,SALEDATE_WeekofMonth,SALEDATE_MonthName,Sum_SOLD_UNITS,Sum_REGISTERED_UNITS,LANE,VRLOCATION,Avg_BUYFEE-FINAL,BUYFEE_CHANGE_FLAG,BUYFEE_CHANGE_STATUS,BUYFEE_CHANGE_DATE,PREVIOUS_BUYFEE,BUYFEE_CHANGE_AMOUNT,Avg_Vehicle_year,Max_Make,Max_Model,WEEKLY_CONVERSION
0,2023,2,August,1,1,YS,BIRMINGHAM,440.0,0,No Change,NaN,440.0,0.0,2020.000000,HYUNDAI,Elantra,1.0
1,2022,3,December,3,3,D3,KALAMAZOO,145.0,0,No Change,NaN,145.0,0.0,2007.333333,TOYOTA,SIENNA,1.0
2,2025,3,March,1,1,F,TULSA,740.0,0,No Change,NaN,740.0,0.0,2023.000000,TOYOTA,Tacoma,1.0
3,2023,3,July,1,1,4A,PHOENIX,220.0,0,No Change,NaN,220.0,0.0,2015.000000,MITSUBISHI,Mirage,1.0
4,2024,3,January,1,1,E,HARRISBURG,485.0,0,No Change,NaN,485.0,0.0,2015.000000,FORD,Transit Cargo Van,1.0


In [3]:
# ==============================
# 3. VALIDATE REQUIRED COLUMNS
# ==============================
required_base = GROUP_COLS + ["Sum_SOLD_UNITS", "Sum_REGISTERED_UNITS"]
missing = [c for c in required_base if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Convert unit columns safely to numeric.
for c in ["Sum_SOLD_UNITS", "Sum_REGISTERED_UNITS"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Required columns validated.")


Required columns validated.


In [4]:
# ==============================
# 4. CREATE / DETECT A WEEK SORT KEY
# ==============================
# Best case: the source already has an actual weekly date.
DATE_CANDIDATES = [
    "Week_Start", "WEEK_START", "WeekStart",
    "SALEDATE_Week_Start", "VRSALEDATE", "SALEDATE"
]

date_col = next((c for c in DATE_CANDIDATES if c in df.columns), None)

if date_col is not None:
    # Handles standard date strings and YYYYMMDD-style values.
    raw = df[date_col].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    parsed = pd.to_datetime(raw, errors="coerce")
    yyyymmdd_mask = raw.str.fullmatch(r"\d{8}", na=False)
    parsed.loc[yyyymmdd_mask] = pd.to_datetime(raw.loc[yyyymmdd_mask], format="%Y%m%d", errors="coerce")

    # Normalize any transaction/date value to Monday of its week.
    df["_Week_Sort"] = parsed - pd.to_timedelta(parsed.dt.weekday, unit="D")
    print(f"Using '{date_col}' to order weeks.")

elif all(c in df.columns for c in ["SALEDATE_Year", "SALEDATE_MonthName", "SALEDATE_WeekofMonth"]):
    # Fallback for files like the supplied sample, which do not contain an actual Week_Start.
    # Creates a deterministic weekly sort key from Year + Month + Week-of-Month.
    year = pd.to_numeric(df["SALEDATE_Year"], errors="coerce").astype("Int64")
    wom = pd.to_numeric(df["SALEDATE_WeekofMonth"], errors="coerce").astype("Int64")
    month_num = pd.to_datetime(df["SALEDATE_MonthName"].astype(str).str[:3], format="%b", errors="coerce").dt.month

    first_day = pd.to_datetime(
        year.astype(str) + "-" + month_num.astype("Int64").astype(str) + "-01",
        errors="coerce"
    )
    df["_Week_Sort"] = first_day + pd.to_timedelta((wom - 1) * 7, unit="D")
    print("No actual week-date column found; using Year + Month + Week-of-Month as the chronological key.")
    print("For exact week-over-week matching, adding a true Week_Start column to the source is recommended.")
else:
    raise ValueError(
        "Cannot determine weekly order. Add a Week_Start/date column, or include "
        "SALEDATE_Year + SALEDATE_MonthName + SALEDATE_WeekofMonth."
    )

if df["_Week_Sort"].isna().any():
    print(f"WARNING: {df['_Week_Sort'].isna().sum():,} rows could not be assigned a weekly sort date.")

display(df[GROUP_COLS + ["_Week_Sort"]].head())


No actual week-date column found; using Year + Month + Week-of-Month as the chronological key.
For exact week-over-week matching, adding a true Week_Start column to the source is recommended.


,VRLOCATION,LANE,_Week_Sort
0,BIRMINGHAM,YS,2023-08-08
1,KALAMAZOO,D3,2022-12-15
2,TULSA,F,2025-03-15
3,PHOENIX,4A,2023-07-15
4,HARRISBURG,E,2024-01-15


In [5]:
# ==============================
# 5. CALCULATE WEEKLY CONVERSION
# ==============================
# Recalculate from sold / registered units so the logic is consistent.
# Division by zero becomes NaN rather than infinity/error.
registered = df["Sum_REGISTERED_UNITS"].replace(0, np.nan)
df["Weekly_Conversion"] = df["Sum_SOLD_UNITS"] / registered
df["Weekly_Conversion_Pct"] = df["Weekly_Conversion"] * 100

# Sort before any lag/rolling calculations.
df = df.sort_values(GROUP_COLS + ["_Week_Sort"], kind="stable").reset_index(drop=True)

print(df[["Sum_SOLD_UNITS", "Sum_REGISTERED_UNITS", "Weekly_Conversion", "Weekly_Conversion_Pct"]].head())


   Sum_SOLD_UNITS  Sum_REGISTERED_UNITS  Weekly_Conversion  \
0              53                    72           0.736111   
1              36                    52           0.692308   
2              59                    75           0.786667   
3              28                    50           0.560000   
4              50                    70           0.714286   

   Weekly_Conversion_Pct  
0              73.611111  
1              69.230769  
2              78.666667  
3              56.000000  
4              71.428571  


In [6]:
# ==============================
# 6. PREVIOUS WEEK + WOW CHANGE
# ==============================
g = df.groupby(GROUP_COLS, dropna=False, sort=False)

# Previous observed weekly record within Location × Lane.
df["Previous_Week_Conversion"] = g["Weekly_Conversion"].shift(1)

# Change in conversion RATE (e.g. 0.05 = +5 percentage points).
df["WoW_Conversion_Change"] = (
    df["Weekly_Conversion"] - df["Previous_Week_Conversion"]
)


In [7]:
# ==============================
# 7. PRIOR 4 / 8 / 12 WEEK FEATURES
# ==============================
# Shift first, then roll: this guarantees the current week's conversion is excluded.
def add_prior_rolling_features(data, group_cols, value_col, windows=(4, 8, 12), min_periods=1):
    out = data.copy()
    shifted = out.groupby(group_cols, dropna=False, sort=False)[value_col].shift(1)

    # Temporary shifted series lets groupby.rolling stay aligned with each segment.
    out["_Prior_Conversion"] = shifted

    for w in windows:
        roll = (
            out.groupby(group_cols, dropna=False, sort=False)["_Prior_Conversion"]
               .rolling(window=w, min_periods=min_periods)
        )
        mean_vals = roll.mean().reset_index(level=group_cols, drop=True)
        std_vals = roll.std(ddof=0).reset_index(level=group_cols, drop=True)

        out[f"Prior_{w}wk_Conversion"] = mean_vals
        out[f"Prior_{w}wk_Conversion_Volatility"] = std_vals

    return out.drop(columns=["_Prior_Conversion"])

df = add_prior_rolling_features(
    df, GROUP_COLS, "Weekly_Conversion", windows=(4, 8, 12), min_periods=MIN_PERIODS
)


In [9]:
# ==============================
# 8. CONVERSION TREND FLAG
# ==============================
def classify_trend(change, threshold=0.05):
    if pd.isna(change):
        return "No Prior Week"
    if change > threshold:
        return "Increasing > 5 Points"
    if change < -threshold:
        return "Decreasing > 5 Points"
    return "Stable Within +/- 5 Points"

df["Conversion_Trend_Flag"] = df["WoW_Conversion_Change"].apply(
    lambda x: classify_trend(x, TREND_THRESHOLD)
)


In [10]:
# ==============================
# 9. QA CHECKS
# ==============================
new_cols = [
    "Weekly_Conversion",
    "Weekly_Conversion_Pct",
    "Previous_Week_Conversion",
    "WoW_Conversion_Change",
    "Prior_4wk_Conversion",
    "Prior_8wk_Conversion",
    "Prior_12wk_Conversion",
    "Prior_4wk_Conversion_Volatility",
    "Prior_8wk_Conversion_Volatility",
    "Prior_12wk_Conversion_Volatility",
    "Conversion_Trend_Flag",
]

# Sanity checks
invalid_conversion = df["Weekly_Conversion"].dropna().loc[lambda s: (s < 0) | (s > 1)]
if len(invalid_conversion):
    print(f"WARNING: {len(invalid_conversion):,} rows have conversion outside 0-1. Check source counts.")

dupe_count = df.duplicated(GROUP_COLS + ["_Week_Sort"], keep=False).sum()
if dupe_count:
    print(f"WARNING: {dupe_count:,} rows share the same Location × Lane × week key.")
else:
    print("No duplicate Location × Lane × week rows found.")

print("\nTrend flag counts:")
print(df["Conversion_Trend_Flag"].value_counts(dropna=False))

display(df[GROUP_COLS + ["_Week_Sort"] + new_cols].head(15))



Trend flag counts:
Conversion_Trend_Flag
Stable Within +/- 5 Points    1278980
Decreasing > 5 Points          216979
Increasing > 5 Points          216768
No Prior Week                    1428
Name: count, dtype: int64


,VRLOCATION,LANE,_Week_Sort,Weekly_Conversion,Weekly_Conversion_Pct,Previous_Week_Conversion,WoW_Conversion_Change,Prior_4wk_Conversion,Prior_8wk_Conversion,Prior_12wk_Conversion,Prior_4wk_Conversion_Volatility,Prior_8wk_Conversion_Volatility,Prior_12wk_Conversion_Volatility,Conversion_Trend_Flag
0,ALBANY,A,2022-01-08,0.736111,73.611111,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No Prior Week
1,ALBANY,A,2022-01-15,0.692308,69.230769,0.736111,-0.043803,0.736111,0.736111,0.736111,0.000000,0.000000,0.000000,Stable Within +/- 5 Points
2,ALBANY,A,2022-01-22,0.786667,78.666667,0.692308,0.094359,0.714209,0.714209,0.714209,0.021902,0.021902,0.021902,Increasing > 5 Points
3,ALBANY,A,2022-02-01,0.560000,56.000000,0.786667,-0.226667,0.738362,0.738362,0.738362,0.038555,0.038555,0.038555,Decreasing > 5 Points
4,ALBANY,A,2022-02-08,0.714286,71.428571,0.560000,0.154286,0.693771,0.693771,0.693771,0.084141,0.084141,0.084141,Increasing > 5 Points
5,ALBANY,A,2022-02-15,0.600000,60.000000,0.714286,-0.114286,0.688315,0.697874,0.697874,0.081897,0.075704,0.075704,Decreasing > 5 Points
6,ALBANY,A,2022-02-22,0.722222,72.222222,0.600000,0.122222,0.665238,0.681562,0.681562,0.090113,0.078144,0.078144,Increasing > 5 Points
7,ALBANY,A,2022-03-01,0.630769,63.076923,0.722222,-0.091453,0.649127,0.687370,0.687370,0.070615,0.073733,0.073733,Decreasing > 5 Points
8,ALBANY,A,2022-03-08,0.848837,84.883721,0.630769,0.218068,0.666819,0.680295,0.680295,0.052647,0.071466,0.071466,Increasing > 5 Points
9,ALBANY,A,2022-03-15,0.687500,68.750000,0.848837,-0.161337,0.700457,0.694386,0.699022,0.096745,0.089834,0.085706,Decreasing > 5 Points


In [11]:
# ==============================
# 10. EXPORT CSV + EXCEL
# ==============================
# Internal helper is not needed in the final files.
final_df = df.drop(columns=["_Week_Sort"]).copy()

# Put the requested detailed conversion fields at the end while preserving every source column.
# If the input already had one of these fields, this uses the recalculated version without duplicating it.
base_cols = [c for c in final_df.columns if c not in new_cols]
final_df = final_df[base_cols + new_cols]

final_df.to_csv(OUTPUT_CSV, index=False)

try:
    final_df.to_excel(OUTPUT_XLSX, index=False, engine="openpyxl")
    print(f"Saved Excel: {OUTPUT_XLSX.resolve()}")
except Exception as exc:
    print(f"Excel export skipped: {exc}")

print(f"Saved CSV:   {OUTPUT_CSV.resolve()}")
print(f"Final rows: {len(final_df):,} | Final columns: {len(final_df.columns):,}")
display(final_df.head())


Excel export skipped: This sheet is too large! Your sheet size is: 1714155, 28 Max sheet size is: 1048576, 16384
Saved CSV:   C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\conversion_data_detailed.csv
Final rows: 1,714,155 | Final columns: 28


,SALEDATE_Year,SALEDATE_WeekofMonth,SALEDATE_MonthName,Sum_SOLD_UNITS,Sum_REGISTERED_UNITS,LANE,VRLOCATION,Avg_BUYFEE-FINAL,BUYFEE_CHANGE_FLAG,BUYFEE_CHANGE_STATUS,...,Weekly_Conversion_Pct,Previous_Week_Conversion,WoW_Conversion_Change,Prior_4wk_Conversion,Prior_8wk_Conversion,Prior_12wk_Conversion,Prior_4wk_Conversion_Volatility,Prior_8wk_Conversion_Volatility,Prior_12wk_Conversion_Volatility,Conversion_Trend_Flag
0,2022,2,January,53,72,A,ALBANY,NaN,0,No Change,...,73.611111,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No Prior Week
1,2022,3,January,36,52,A,ALBANY,NaN,0,No Change,...,69.230769,0.736111,-0.043803,0.736111,0.736111,0.736111,0.000000,0.000000,0.000000,Stable Within +/- 5 Points
2,2022,4,January,59,75,A,ALBANY,NaN,0,No Change,...,78.666667,0.692308,0.094359,0.714209,0.714209,0.714209,0.021902,0.021902,0.021902,Increasing > 5 Points
3,2022,1,February,28,50,A,ALBANY,NaN,0,No Change,...,56.000000,0.786667,-0.226667,0.738362,0.738362,0.738362,0.038555,0.038555,0.038555,Decreasing > 5 Points
4,2022,2,February,50,70,A,ALBANY,NaN,0,No Change,...,71.428571,0.560000,0.154286,0.693771,0.693771,0.693771,0.084141,0.084141,0.084141,Increasing > 5 Points
